In [ ]:
import sys
import os
from dotenv import load_dotenv
load_dotenv()

sys.path.append(".")
sys.path.append("..")

from comet_ml import API

from training.training_pipeline import (
    extract_feature_vectors,
    normalize_by_max_per_level,
    get_valid_index,
)
from torch.utils.data import DataLoader
from pathlib import Path
from lib.wpt_utils import load_wavelets
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import pandas as pd
import scienceplots
from training.models.ae_regularized import VAERegularized
import torch

In [ ]:
plt.style.use(["science", "ieee", "grid"])

In [ ]:
COMET_API_KEY = os.environ.get("COMET_API_KEY")

# Raw data

In [ ]:
SEQ_1_PATH = Path("../data/wpts/sequence_1_db35")
SEQ_2_PATH = Path("../data/wpts/sequence_2_db35")
SEQ_3_PATH = Path("../data/wpts/sequence_3_db35")

data_train = load_wavelets(SEQ_2_PATH)
data_valid = load_wavelets(SEQ_1_PATH)

In [ ]:
data_train = [d for i, d in enumerate(data_train) if i > 450]

# Artifacts

Versions:

## KL loss
- 0.1.0 (principal_cottage_4699): lowest KL loss [bad]
- 0.1.1 (external_fort_7329): 4th lowest KL loss, beta=1.0 [bad]
- 0.1.2 (hurt_bobolink_510): lowest KL where beta < 1 [bad] <- good for text

## Dist loss
- 1.2.0 (plain_anglerfish_486): lowest dist loss [ok] but bad rec

## Pearson
- 2.1.0 (skilled_nectar_3123): best avg pearson with latent 2 [bad]
- 2.2.0 (vast_sprout_7617): best max pearson with lowest beta [ok] <-> cable 4 is different
- 2.3.0 (wandering_mammal_6697): best overall max pearson [ok]
- 3.0.0 (surprised_duck_9569): best min pearson with low rec loss [medium]
- 3.1.0 (smiling_salak_4454) : second best min pearson w/ low rec loss, higher latent dim [bad]
- 3.2.0 (loud_tiglon_2638): second best min pearson [ok]
- 3.3.0 (subsequent_sapwood_5305): low delta and high max [bad]
- 3.4.0 (passive_bay_4333): low delta [bad]
- 3.5.0 (dull_saddle_4335): high min_pearson w/ criteria [ok]

## Rec loss
- 4.1.0 (colorful_moss_1642): best reconstruction <- good for text
- 4.2.0 (territorial_ice_8192): best recconstruction with low KL, beta=-2 [medium] [cable 3 bad]

In [ ]:
COMET_WORKSPACE = os.environ.get("COMET_WORKSPACE")
COMET_PROJECT = os.environ.get("COMET_PROJECT_VAE")
COMET_MODEL_NAME = os.environ.get("COMET_MODEL_NAME")
COMET_MODEL_VERSION = os.environ.get("COMET_MODEL_VERSION")

api = API(api_key=COMET_API_KEY)
model = api.get_model(COMET_WORKSPACE, COMET_MODEL_NAME)
filename = model.get_assets(COMET_MODEL_VERSION)[0]["fileName"]
model_path = Path("../training/models/ckpt/")

model.download(version=COMET_MODEL_VERSION, output_folder=model_path)

In [ ]:
# Get the experiment associated with the model asset to retrieve logged hyperparameters
experiment_key = model.get_assets(COMET_MODEL_VERSION)[0]["experimentKey"]
experiment = api.get_experiment(COMET_WORKSPACE, COMET_PROJECT, experiment_key)
experiment_name = experiment.get_metadata()["experimentName"]
params_list = experiment.get_parameters_summary()
params = {p["name"]: p for p in params_list}

lr = float(params["lr"]["valueMax"])
latent_dim = int(params["latent_dim"]["valueMax"])
hidden_dim = int(params["hidden_dim"]["valueMax"])
num_layers = int(params["num_layers"]["valueMax"])
activation = params["activation"]["valueMax"]
kl_weight = float(params["kl_weight"]["valueMax"])
beta = float(params["beta"]["valueMax"])
bandwidth = float(params["bandwidth"]["valueMax"])
min_level = int(params["min_level"]["valueMax"])
max_level = int(params["max_level"]["valueMax"])
f_min = float(params["f_min"]["valueMax"])

In [ ]:
metrics = experiment.get_metrics()

In [ ]:
keys_of_interest = [
    "train_dist_loss",
    "train_rec_loss",
    "train_kl_loss",
    "val_dist_loss",
    "val_rec_loss",    "val_kl_loss",
    "train_loss",
    "val_loss",
]
selected = [m for m in metrics if m["metricName"] in keys_of_interest]

In [ ]:
training_metrics = pd.DataFrame(selected)

In [ ]:
print(experiment_name)

# Training parameters

In [ ]:
training_metrics["type"] = training_metrics["metricName"].str.split("_").str[0]
training_metrics["metricValue"] = training_metrics["metricValue"].astype(float)
training_metrics["metric_name_clean"] = training_metrics["metricName"].str.split("_").str[1]

In [ ]:
metric_dic = {
    "dist": "Distance Loss",
    "rec": "Reconstruction Loss",
    "kl": "KL Loss",
    "loss": "Total Loss",
}

metric_names = [name for name in training_metrics["metric_name_clean"].unique()]

fig, axes = plt.subplots(2, 2, figsize=(4, 3), sharex=True)
axes = axes.flatten()
for ax, metric in zip(axes, metric_names):
    subset = training_metrics[training_metrics["metric_name_clean"] == metric].sort_values(by=["type", "epoch"])
    for t in subset["type"].unique():
        metric_type_subset = subset[subset["type"] == t]
        ax.plot(metric_type_subset["epoch"], metric_type_subset["metricValue"], label=t)
    # ax.set_title(metric_names[metric])
    ax.set_xlabel("Epoch")
    ax.set_ylabel(metric_dic[metric])
    ax.legend(["Train", "Validation"], fontsize='small')
    ax.grid(True)

plt.tight_layout()
plt.savefig("figures_vae_results/loss_train_valid.png", dpi=600)
plt.show()

# Data

In [ ]:
f_max = f_min + bandwidth

In [ ]:
valid_index = get_valid_index(
    f_min=f_min, f_max=f_max, min_level=min_level, max_level=max_level, fs=256
)

In [ ]:
def prepare_data_and_loader(data, valid_index, max_level, batch_size=32):
    x_tensor, t_tensor, dates = extract_feature_vectors(data)
    xn = normalize_by_max_per_level(x_tensor, max_level=max_level + 1)
    valid_index_flat = [item for sublist in valid_index.values() for item in sublist]

    temp = np.zeros((xn.shape[0], xn.shape[1], len(valid_index_flat)))
    for cable in [0, 1, 2, 3]:
        temp[:, cable] = xn[:, cable][:, valid_index_flat]
    xn = temp

    dataloader = DataLoader(xn, batch_size=batch_size, shuffle=False, num_workers=0)
    return xn, t_tensor, dates, dataloader

In [ ]:
x_train, t_train, dates_train, train_dataloader = prepare_data_and_loader(
    data_train, valid_index, max_level - 1
)

x_valid, t_valid, dates_valid, valid_dataloader = prepare_data_and_loader(
    data_valid, valid_index, max_level - 1
)


In [ ]:
del data_train
del data_valid

In [ ]:
data_test = load_wavelets(SEQ_3_PATH)

In [ ]:
x_test, t_test, dates_test, test_dataloader = prepare_data_and_loader(
    data_test, valid_index, max_level - 1
)

In [ ]:
input_dim = x_train.shape[2]
print(input_dim)

# Model instantiation and inference

In [ ]:
from training.evaluate import evaluate_vae

In [ ]:
latent_dim, max_level

In [ ]:
model = VAERegularized.load_from_checkpoint(
    model_path / filename,
    input_dim=input_dim,
    lr=lr,
    latent_dim=latent_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    activation=activation,
    beta=beta,
)
model.eval();

In [ ]:
model.to("cpu");

In [ ]:
x_train, z_train, mse_train, y_train = evaluate_vae(train_dataloader, model)
x_valid, z_valid, mse_valid, y_valid = evaluate_vae(valid_dataloader, model)

In [ ]:
x_test, z_test, mse_test, y_test = evaluate_vae(test_dataloader, model)

In [ ]:
plt.figure(figsize=(5, 8))

k = 1
for cable in [0, 1, 2, 3]:
    ind = np.arange(x_train.shape[-1])

    plt.subplot(4, 2, k)
    c = plt.pcolormesh(ind, t_train[:, cable], x_train[:, cable, :], cmap="Blues")
    plt.xlabel("WPT Index")
    plt.ylabel("Tension (N)")
    plt.title(f"Input - Cable {cable + 1}")
    cbar = plt.colorbar(c, label=rf"$\mathbf{{x}}_{{{cable + 1}}}^T[n]$")
    cbar.set_ticks(np.arange(0, 1.2, 0.2))
    k += 1

    plt.subplot(4, 2, k)
    c = plt.pcolormesh(ind, t_train[:, cable], y_train[:, cable, :], cmap="Blues")
    plt.xlabel("WPT Index")
    plt.ylabel("Tension (N)")
    plt.title(f"Reconstruction - Cable {cable + 1}")
    cbar = plt.colorbar(c, label=rf"$\mathbf{{y}}_{{{cable + 1}}}^T[n]$")
    cbar.set_ticks(np.arange(0, 1.2, 0.2))
    k += 1

plt.tight_layout()
plt.savefig("figures_vae_results/train_reconstruction.png", dpi=600)
plt.show()


In [ ]:
plt.figure(figsize=(5, 8))

k = 1
for cable in [0, 1, 2, 3]:
    ind = np.arange(x_valid.shape[-1])

    plt.subplot(4, 2, k)
    c = plt.pcolormesh(ind, t_valid[:, cable], x_valid[:, cable, :], cmap="Blues")
    plt.xlabel("WPT Index")
    plt.ylabel("Tension (N)")
    plt.title(f"Input - Cable {cable + 1}")
    cbar = plt.colorbar(c, label=rf"$\mathbf{{x}}_{{{cable + 1}}}^V[n]$")
    cbar.set_ticks(np.arange(0, 1.2, 0.2))
    k += 1

    plt.subplot(4, 2, k)
    c = plt.pcolormesh(ind, t_valid[:, cable], y_valid[:, cable, :], cmap="Blues")
    plt.xlabel("WPT Index")
    plt.ylabel("Tension (N)")
    plt.title(f"Reconstruction - Cable {cable + 1}")
    cbar = plt.colorbar(c, label=rf"$\mathbf{{y}}_{{{cable + 1}}}^V[n]$")
    cbar.set_ticks(np.arange(0, 1.2, 0.2))
    k += 1

plt.tight_layout()
plt.savefig("figures_vae_results/valid_reconstruction.png", dpi=600)
plt.show()


In [ ]:
plt.figure(figsize=(5, 8))

k = 1
for cable in [0, 1, 2, 3]:
    ind = np.arange(x_test.shape[-1])

    plt.subplot(4, 2, k)
    c = plt.pcolormesh(ind, t_test[:, cable], x_test[:, cable, :], cmap="Blues")
    plt.xlabel("WPT Index")
    plt.ylabel("Tension (N)")
    plt.title(f"Input - Cable {cable + 1}")
    cbar = plt.colorbar(c, label=rf"$\mathbf{{x}}_{{{cable + 1}}}^{{Test}}[n]$")
    cbar.set_ticks(np.arange(0, 1.2, 0.2))
    k += 1

    plt.subplot(4, 2, k)
    c = plt.pcolormesh(ind, t_test[:, cable], y_test[:, cable, :], cmap="Blues")
    plt.xlabel("WPT Index")
    plt.ylabel("Tension (N)")
    plt.title(f"Reconstruction - Cable {cable + 1}")
    cbar = plt.colorbar(c, label=rf"$\mathbf{{y}}_{{{cable + 1}}}^{{Test}}[n]$")
    cbar.set_ticks(np.arange(0, 1.2, 0.2))
    k += 1

plt.tight_layout()
plt.savefig("figures_vae_results/test_reconstruction.png", dpi=600)
plt.show()


In [ ]:
rec_loss = torch.nn.BCEWithLogitsLoss(reduction="mean")

In [ ]:
train_rec_loss = np.zeros((x_train.shape[0], x_train.shape[1]))
for k in range(x_train.shape[0]):
    for cable in [0,1,2,3]:
        train_rec_loss[k,cable] = rec_loss(torch.Tensor(x_train[k,cable]), torch.Tensor(y_train[k,cable]))

valid_rec_loss = np.zeros((x_valid.shape[0], x_valid.shape[1]))
for k in range(x_valid.shape[0]):
    for cable in [0,1,2,3]:
        valid_rec_loss[k,cable] = rec_loss(torch.Tensor(x_valid[k,cable]), torch.Tensor(y_valid[k,cable]))

In [ ]:
test_rec_loss = np.zeros((x_test.shape[0], x_test.shape[1]))
for k in range(x_test.shape[0]):
    for cable in [0,1,2,3]:
        test_rec_loss[k,cable] = rec_loss(torch.Tensor(x_test[k,cable]), torch.Tensor(y_test[k,cable]))

In [ ]:
plt.figure(figsize=(4,4))
y_min = np.min([train_rec_loss.min(), valid_rec_loss.min()]) * 0.99
y_max = np.max([train_rec_loss.max(), valid_rec_loss.max()]) * 1.01

for cable in range(4):
    plt.subplot(2,2,cable+1)

    plt.scatter([], [], color="k", s=3)
    plt.scatter([],[],color="red", marker="d", s=3)
    if cable != 3:
        plt.legend(["Train", "Validation"], fontsize=7, framealpha=0)
    else:
        plt.legend(["Train", "Validation"], fontsize=7, framealpha=0, loc="lower left")
    plt.scatter(t_train[:,cable], train_rec_loss[:,cable], s=0.5, marker="o")
    plt.scatter(t_valid[:,cable], valid_rec_loss[:,cable], s=0.5, marker="x")
    plt.ylabel("Reconstruction Loss")
    plt.xlabel("Tension (N)")
    plt.title(f"Cable {cable + 1}")
    plt.ylim([y_min, y_max])
    
plt.tight_layout()
plt.savefig("figures_vae_results/loss_per_sample.png", dpi=400)

In [ ]:
plt.figure(figsize=(4,4))
y_min = np.min([train_rec_loss.min(), valid_rec_loss.min()]) * 0.99
y_max = np.max([train_rec_loss.max(), valid_rec_loss.max()]) * 1.01

for cable in range(4):
    plt.subplot(2,2,cable+1)

    plt.scatter([], [], color="k", s=3)
    plt.scatter([],[],color="blue", marker="x", s=3)
    plt.scatter([],[],color="red", marker="d", s=3)
    if cable != 3:
        plt.legend(["Train", "Validation", "Test"], fontsize=7, framealpha=0)
    else:
        plt.legend(["Train", "Validation", "Test"], fontsize=7, framealpha=0, loc="lower left")
    plt.scatter(t_train[:,cable], train_rec_loss[:,cable], s=0.5, marker="o", alpha=0.1)
    plt.scatter(t_valid[:,cable], valid_rec_loss[:,cable], s=0.5, marker="x", color="blue", alpha=0.1)
    plt.scatter(t_test[:,cable], test_rec_loss[:,cable], s=0.5, marker="d", color="red")
    plt.ylabel("Reconstruction Loss")
    plt.xlabel("Tension (N)")
    plt.title(f"Cable {cable + 1}")
    plt.ylim([y_min, y_max])
    
plt.tight_layout()
plt.savefig("figures_vae_results/loss_per_sample_test.png", dpi=400)

# Embedding distances

In [ ]:
days = 10
max_idx = int(days * 24 * 60 / 15)
print(max_idx)
reference_window = np.arange(0, max_idx, 1)

References

In [ ]:
t_train_ref = t_train[reference_window].mean(axis=0)
t_valid_ref = t_valid[reference_window].mean(axis=0)

z_train_ref = z_train[reference_window].mean(axis=0)
z_valid_ref = z_valid[reference_window].mean(axis=0)

In [ ]:
t_test_ref = t_test[reference_window].mean(axis=0)
z_test_ref = z_test[reference_window].mean(axis=0)

Tension drop variables

In [ ]:
t_train_drop = abs(t_train - t_train_ref) / t_train_ref
t_valid_drop = abs(t_valid - t_valid_ref) / t_valid_ref

In [ ]:
t_test_drop = abs(t_test - t_test_ref) / t_test_ref

In [ ]:
plt.figure(figsize=(5, 4))

z_min = np.min(z_train) * 1.05
z_max = np.max(z_train) * 1.05

colors = ["black", "red", "blue", "green", "orange", "magenta", "purple", "brown"]

for cable in [0, 1, 2, 3]:
    plt.subplot(2, 2, cable + 1)
    plt.title(f"Cable {cable + 1}")
    for k in range(z_train.shape[2]):
        plt.scatter(t_train_drop[:, cable], z_train[:, cable, k], alpha=1, s=0.01, color=colors[k], marker="x")
        plt.ylim(z_min, z_max)
    plt.xlabel(rf"$\bar{{t}}_{cable+1}^T[n]$")
    plt.ylabel(rf"$\mathbf{{z}}_{cable+1}^T[n]$")
    plt.grid(False)

plt.tight_layout()
plt.savefig("figures_vae_results/train_embeddings.png", dpi=600)
plt.show()

In [ ]:
plt.figure(figsize=(5, 4))

z_min = np.min(z_valid) * 1.05
z_max = np.max(z_valid) * 1.05

colors = ["black", "red", "blue", "green", "orange", "magenta", "purple", "brown"]

for cable in [0, 1, 2, 3]:
    plt.subplot(2, 2, cable + 1)
    plt.title(f"Cable {cable + 1}")
    for k in range(z_valid.shape[2]):
        plt.scatter(t_valid_drop[:, cable], z_valid[:, cable, k], alpha=1, s=0.01, color=colors[k], marker="x")
        plt.ylim(z_min, z_max)

    plt.xlabel(rf"$\bar{{t}}_{cable+1}^V[n]$")
    plt.ylabel(rf"$\mathbf{{z}}_{cable+1}^V[n]$")
    plt.grid(False)
plt.tight_layout()
plt.savefig("figures_vae_results/valid_embeddings.png", dpi=600)

plt.show()

In [ ]:
np.round(t_train_ref,1), np.round(t_valid_ref,1)

In [ ]:
np.round(t_test_ref,1)

In [ ]:
train_df = pd.DataFrame(
    {
        "date": dates_train,
        "value": [z_train[i] for i in range(z_train.shape[0])],
        "tension": [t_train_drop[i] for i in range(t_train_drop.shape[0])],
        "tension_unnorm": [t_train[i] for i in range(t_train_drop.shape[0])],
    }
)
train_df = train_df.set_index("date")


valid_df = pd.DataFrame(
    {
        "date": dates_valid,
        "value": [z_valid[i] for i in range(z_valid.shape[0])],
        "tension": [t_valid_drop[i] for i in range(t_valid_drop.shape[0])],
        "tension_unnorm": [t_valid[i] for i in range(t_valid_drop.shape[0])],
    }
)
valid_df = valid_df.set_index("date")

In [ ]:
test_df = pd.DataFrame(
    {
        "date": dates_test,
        "value": [z_test[i] for i in range(z_test.shape[0])],
        "tension": [t_test_drop[i] for i in range(t_test_drop.shape[0])],
        "tension_unnorm": [t_test[i] for i in range(t_test_drop.shape[0])],
    }
)
test_df = test_df.set_index("date")

In [ ]:
train_days_skip = 0
valid_days_skip = 0

# skip first days from training set

train_df_daily = train_df.groupby(train_df.index.date).mean()
train_df_daily = train_df_daily.iloc[train_days_skip:]

z_train_gb = np.array(train_df_daily["value"].tolist())
t_train_gb = np.array(train_df_daily["tension"].tolist())
t_train_unnorm_gb = np.array(train_df_daily["tension_unnorm"].tolist())

# -----

# skip first days from validation set

valid_df_daily = valid_df.groupby(valid_df.index.date).mean()
valid_df_daily = valid_df_daily.iloc[valid_days_skip:]

z_valid_gb = np.array(valid_df_daily["value"].tolist())
t_valid_gb = np.array(valid_df_daily["tension"].tolist())
t_valid_unnorm_gb = np.array(valid_df_daily["tension_unnorm"].tolist())

In [ ]:
test_days_skip = 0

test_df_daily = test_df.groupby(test_df.index.date).mean()
test_df_daily = test_df_daily.iloc[test_days_skip:]

z_test_gb = np.array(test_df_daily["value"].tolist())
t_test_gb = np.array(test_df_daily["tension"].tolist())
t_test_unnorm_gb = np.array(test_df_daily["tension_unnorm"].tolist())

In [ ]:
reference_window_gb = np.arange(0, days, 1)

t_train_gb_ref = t_train_gb[reference_window_gb].mean(axis=0)
t_train_gb_drop = (t_train_gb) / t_train_gb_ref

t_valid_gb_ref = t_valid_gb[reference_window_gb].mean(axis=0)
t_valid_gb_drop = (t_valid_gb) / t_valid_gb_ref

In [ ]:
test_reference_window_gb = np.arange(0, 10, 1)

t_test_gb_ref = t_test_gb[test_reference_window_gb].mean(axis=0)
t_test_gb_drop = (t_test_gb) / t_test_gb_ref

In [ ]:
t_train_gb_drop = 1 - t_train_gb_drop
t_valid_gb_drop = 1 - t_valid_gb_drop

In [ ]:
t_test_gb_drop = 1 - t_test_gb_drop

In [ ]:
from scipy.spatial.distance import mahalanobis, seuclidean

class IndicatorsExtractor:
    def __init__(self, data: np.ndarray, ref_ids: list[int]):
        self.data = data  # np.ndarray of shape (n_samples, 4, n_dim)
        self.ref_data = data[ref_ids]
        self.ref_mean = np.mean(self.ref_data, axis=0)
        # self.ref_mean = np.quantile(self.ref_data, 0.5, axis=0)
        self.ref_std = np.std(self.ref_data, axis=0)
        self.ref_norm = np.linalg.norm(self.ref_mean, axis=-1)

    def extract_l1(self):
        l1_norms = np.linalg.norm(self.data - self.ref_mean, ord=1, axis=-1)
        return l1_norms

    def extract_l2(self):
        l2_norms = np.linalg.norm(self.data - self.ref_mean, ord=2, axis=-1)
        return l2_norms

    def extract_l2_rel(self):
        l2_norms = np.linalg.norm(self.data - self.ref_mean, ord=2, axis=-1)
        l2_rel = l2_norms / self.ref_norm
        return l2_rel

    def extract_cosine_sim(self):
        cos = np.zeros((self.data.shape[0], self.data.shape[1]))
        norms = np.linalg.norm(self.data, axis=-1)

        for i in range(self.data.shape[0]):
            for j in range(self.data.shape[1]):
                cos[i, j] = np.dot(self.data[i, j], self.ref_mean[j]) / (
                    norms[i, j] * self.ref_norm[j]
                )
        return cos

    def extract_mahalanobis(self):
        cov_inv = np.zeros((self.data.shape[1], self.data.shape[2], self.data.shape[2]))
        for j in range(self.data.shape[1]):
            cov = np.cov(self.ref_data[:, j, :].T)
            cov_inv[j] = np.linalg.inv(cov)

        mahal_dist = np.zeros((self.data.shape[0], self.data.shape[1]))
        for i in range(self.data.shape[0]):
            for j in range(self.data.shape[1]):
                mahal_dist[i, j] = mahalanobis(
                    self.data[i, j], self.ref_mean[j], cov_inv[j]
                )
        return mahal_dist


    def compute(self):
        #l1 = self.extract_l1()
        l2 = self.extract_l2_rel()
        #cos = self.extract_cosine_sim()
        l2_dist = np.diff(l2, axis=0, prepend=l2[0:1])

        # return np.stack([l1, l2, cos, sleuc_dist], axis=-1)
        # return np.stack([l2, cos], axis=-1)
        return np.stack([l2], axis=-1)
        # return np.stack([5*np.log(l2)], axis=-1)
        # return np.stack([l2, l1], axis=-1)

In [ ]:
# train_ind_raw = IndicatorsExtractor(z_train, ref_ids=reference_window).compute()
# valid_ind_raw = IndicatorsExtractor(z_valid, ref_ids=reference_window).compute()

In [ ]:
train_ind = IndicatorsExtractor(z_train_gb, ref_ids=reference_window_gb).compute()
valid_ind = IndicatorsExtractor(z_valid_gb, ref_ids=reference_window_gb).compute()

In [ ]:
test_ind = IndicatorsExtractor(z_test_gb, ref_ids=test_reference_window_gb).compute()

Train and valid indicators per avg tension

In [ ]:
plt.figure(figsize=(4,2))

ind = 0

y_max = 2.5
x_max = 0.55

plt.subplot(1,2,1)
min_y = train_ind[:,:,ind].min() * 0.95
max_y = train_ind[:,:,ind].max() * 1.05
for cable in range(4):
    plt.plot(t_train_gb[:,cable], train_ind[:,cable,ind], alpha=1, linewidth=0.9, linestyle="-")
plt.title("Train set")
plt.ylabel("$h_j[n]$")
plt.xlabel(r"$\bar{t}_j[n]$")
plt.legend(["$j=1$", "$j=2$", "$j=3$", "$j=4$"], fontsize=5, framealpha=1, loc="upper right")
plt.xticks([0, 0.1, 0.2, 0.3, 0.4, 0.5])
plt.ylim([0, y_max])
plt.xlim([0, x_max])

plt.subplot(1,2,2)
min_y = valid_ind[:,:,ind].min() * 0.95
max_y = valid_ind[:,:,ind].max() * 1.05
for cable in range(4):
    plt.plot(t_valid_gb[:,cable], valid_ind[:,cable,ind], alpha=1, linewidth=0.9, linestyle="-")
plt.title("Validation set")
plt.xlabel(r"$\bar{t}_j[n]$")
plt.ylabel("$h_j[n]$")
plt.legend(["$j=1$", "$j=2$", "$j=3$", "$j=4$"], fontsize=5, framealpha=1, loc="upper right")
plt.xlim([0, x_max])
plt.xticks([0, 0.1, 0.2, 0.3, 0.4, 0.5])
plt.ylim([0, y_max])
plt.tight_layout()
plt.savefig("figures_vae_results/indicator_vs_tension_per_set.png", dpi=400)

In [ ]:
plt.figure(figsize=(2,4))

ind = 0

y_max = 2.5
x_max = 0.55

plt.subplot(2,1,1)
min_y = train_ind[:,:,ind].min() * 0.95
max_y = train_ind[:,:,ind].max() * 1.05
for cable in range(4):
    plt.plot(t_train_gb[:,cable], train_ind[:,cable,ind], alpha=1, linewidth=0.9, linestyle="-")
plt.title("Train set")
plt.ylabel("$h_j[n]$")
plt.xlabel(r"$\bar{t}_j[n]$")
plt.legend(["$j=1$", "$j=2$", "$j=3$", "$j=4$"], fontsize=5, framealpha=1, loc="upper right")
plt.xticks([0, 0.1, 0.2, 0.3, 0.4, 0.5])
plt.ylim([0, y_max])
plt.xlim([0, x_max])

plt.subplot(2,1,2)
min_y = valid_ind[:,:,ind].min() * 0.95
max_y = valid_ind[:,:,ind].max() * 1.05
for cable in range(4):
    plt.plot(t_valid_gb[:,cable], valid_ind[:,cable,ind], alpha=1, linewidth=0.9, linestyle="-")
plt.title("Validation set")
plt.xlabel(r"$\bar{t}_j[n]$")
plt.ylabel("$h_j[n]$")
plt.legend(["$j=1$", "$j=2$", "$j=3$", "$j=4$"], fontsize=5, framealpha=1, loc="upper right")
plt.xlim([0, x_max])
plt.xticks([0, 0.1, 0.2, 0.3, 0.4, 0.5])
plt.ylim([0, y_max])
plt.tight_layout()
plt.savefig("figures_vae_results/indicator_vs_tension_per_set_vert.png", dpi=400)

In [ ]:
plt.figure(figsize=(4,3))

ind = 0

y_max = 2.0
# x_max = 0.45

min_y = test_ind[:,:,ind].min() * 0.95
max_y = test_ind[:,:,ind].max() * 1.05
for cable in range(4):
    plt.plot(test_ind[:,cable,ind], alpha=1, linewidth=0.9, linestyle="-")
plt.title("Test set")
plt.xlabel(r"$n$")
plt.ylabel("$h_j[n]$")
plt.legend(["$j=1$", "$j=2$", "$j=3$", "$j=4$"], fontsize=5, framealpha=1, loc="upper right")
# plt.xlim([0, x_max])
# plt.xticks([0, 0.1, 0.2, 0.3, 0.4])
plt.ylim([0, y_max])
plt.tight_layout()
plt.savefig("figures_vae_results/indicator_vs_tension_test.png", dpi=400)

In [ ]:
plt.figure(figsize=(4,3))

ind = 0

y_max = 2.5
x_max = 0.5

min_y = train_ind[:,:,ind].min() * 0.95
max_y = train_ind[:,:,ind].max() * 1.05
for cable in range(4):
    plt.subplot(2,2,cable+1)
    plt.plot(t_train_gb[:,cable], train_ind[:,cable,ind], alpha=1, linewidth=0.9, linestyle="-")
    plt.plot(t_valid_gb[:,cable], valid_ind[:,cable,ind], alpha=1, linewidth=0.9, linestyle="-", color="blue")
    plt.ylabel(r"$h_{%d}[n]$" % (cable + 1))
    plt.xlabel(r"$\bar{t}_{%d}[n]$" % (cable + 1))
    plt.legend(["Train", "Validation"], fontsize=5, framealpha=1, loc="upper left")
    plt.xticks([0, 0.1, 0.2, 0.3, 0.4, 0.5])
plt.ylim([0, y_max])
plt.xlim([0, x_max])
plt.tight_layout()

plt.savefig("figures_vae_results/indicator_vs_tension_per_cable.png", dpi=400)

In [ ]:
plt.figure(figsize=(3,4))

ind = 0

y_max = 2.5
x_max = 0.5

min_y = train_ind[:,:,ind].min() * 0.95
max_y = train_ind[:,:,ind].max() * 1.05

colors = ["black", "red", "blue", "green", "orange", "magenta", "purple", "brown"]

plt.subplot(2,1,1)
plt.title("Training")
for cable in range(4):    
    plt.plot(t_train_unnorm_gb[:,cable], train_ind[:,cable,ind], alpha=1, linewidth=0.9, linestyle="-", color=colors[cable])
    # plt.plot(t_valid_unnorm_gb[:,cable], valid_ind[:,cable,ind], alpha=1, linewidth=0.9, linestyle="--", color=colors[cable], marker="v", markersize=2)
    plt.ylabel(r"$h_j[n]$")
    plt.xlabel("$t_j[n]$ (N)")
plt.legend(["$j=1$", "$j=2$", "$j=3$", "$j=4$"], fontsize=5, framealpha=1, loc="upper right")

plt.subplot(2,1,2)
plt.title("Validation")
for cable in range(4):
    # plt.plot(t_train_unnorm_gb[:,cable], train_ind[:,cable,ind], alpha=1, linewidth=0.9, linestyle="-", color=colors[cable])
    plt.plot(t_valid_unnorm_gb[:,cable], valid_ind[:,cable,ind], alpha=1, linewidth=0.9, linestyle="-", color=colors[cable])
    plt.ylabel(r"$h_j[n]$")
    plt.xlabel("$t_j[n]$ (N)")
plt.legend(["$j=1$", "$j=2$", "$j=3$", "$j=4$"], fontsize=5, framealpha=1, loc="upper right")

plt.tight_layout()

plt.savefig("figures_vae_results/indicator_vs_tension_per_cable_unnormalized.png", dpi=400)

In [ ]:
plt.figure(figsize=(3,4))

ind = 0

y_max = 2.5
x_max = 0.5

min_y = train_ind[:,:,ind].min() * 0.95
max_y = train_ind[:,:,ind].max() * 1.05

colors = ["black", "red", "blue", "green", "orange", "magenta", "purple", "brown"]

plt.subplot(2,1,1)
plt.title("Training")
for cable in range(4):    
    plt.plot(t_train_gb[:,cable], train_ind[:,cable,ind], alpha=1, linewidth=0.9, linestyle="-", color=colors[cable])
    plt.ylabel(r"$h_j[n]$")
    plt.xlabel(r"$\bar{t}_j[n]$")
plt.legend(["$j=1$", "$j=2$", "$j=3$", "$j=4$"], fontsize=5, framealpha=1, loc="upper left")

plt.subplot(2,1,2)
plt.title("Validation")
for cable in range(4):
    plt.plot(t_valid_gb[:,cable], valid_ind[:,cable,ind], alpha=1, linewidth=0.9, linestyle="-", color=colors[cable])
    plt.ylabel(r"$h_j[n]$")
    plt.xlabel(r"$\bar{t}_j[n]$")
plt.legend(["$j=1$", "$j=2$", "$j=3$", "$j=4$"], fontsize=5, framealpha=1, loc="upper left")

plt.tight_layout()

plt.savefig("figures_vae_results/indicator_vs_tension_per_cable_normalized.png", dpi=400)

In [ ]:
plt.figure(figsize=(4,3))

ind = 0

y_max = 2.5
x_max = 0.5

min_y = train_ind[:,:,ind].min() * 0.95
max_y = train_ind[:,:,ind].max() * 1.05
for cable in range(4):
    plt.subplot(2,2,cable+1)
    plt.plot(t_train_gb[:,cable], train_ind[:,cable,ind], alpha=0.2, linewidth=0.9, linestyle="-")
    plt.plot(t_valid_gb[:,cable], valid_ind[:,cable,ind], alpha=0.2, linewidth=0.9, linestyle="-", color="blue")
    plt.plot(t_test_gb[:,cable], test_ind[:,cable,ind], alpha=1, linewidth=0.9, linestyle="-", color="red")
    plt.ylabel(r"$h_{%d}[n]$" % (cable + 1))
    plt.xlabel(r"$\bar{t}_{%d}[n]$" % (cable + 1))
    if cable != 3:
        plt.legend(["Train", "Validation", "Test"], fontsize=5, framealpha=1, loc="lower right")
    else:
        plt.legend(["Train", "Validation", "Test"], fontsize=5, framealpha=1, loc="upper right")
    plt.xticks([0, 0.1, 0.2, 0.3, 0.4, 0.5])
plt.ylim([0, y_max])
plt.xlim([0, x_max])
plt.tight_layout()

plt.savefig("figures_vae_results/indicator_vs_tension_per_cable_test.png", dpi=400)

Train and valid indicators per sample

# HMM

In [ ]:
from hmmlearn.hmm import GaussianHMM, GMMHMM
from sklearn.mixture import GaussianMixture
from scipy.special import logsumexp, softmax

class HMMConditionIndicator():
    def __init__(self, n_states: int, p_self: float = 0.8, n_indicators: int = 4):
        self.n_states = n_states
        self.p_self = p_self
        self.n_indicators = n_indicators
        self.hmm_list = [None] * n_indicators

    def left_to_right_transition_matrix(self):
        """
        Creates a left-to-right transition matrix for an HMM with `n_states` states.
        Each state can only stay in itself (with p_self probability) or go to the next state (with 1-p_self probability),
        except the last state, which is absorbing.

        Args:
            n_states (int): Number of states in the model.
            p_self (float): Probability of staying in the same state. Probability of moving to next: (1 - p_self).

        Returns:
            np.ndarray: Transition matrix of shape (n_states, n_states).
        """
        transmat = np.zeros((self.n_states, self.n_states))
        for i in range(self.n_states):
            if i < self.n_states - 1:
                transmat[i, i] = self.p_self
                transmat[i, i + 1] = 1 - self.p_self
            else:
                transmat[i, i] = 1.0
        return transmat


    def instantiate(self) -> GaussianHMM:
        hmm_model = GaussianHMM(
            n_components=self.n_states,
            covariance_type="diag",
            n_iter=100,
            random_state=42,
            params="stmc",
            init_params="",
        )

        hmm_model.transmat_ = self.left_to_right_transition_matrix()
        hmm_model.startprob_ = np.zeros(self.n_states)
        hmm_model.startprob_[0] = 1.0

        return hmm_model
    
    def score(self, X: np.ndarray) -> np.ndarray:
        n_cables = X.shape[1]
        metrics = np.zeros((3, n_cables, self.n_indicators))
        for cable in range(n_cables):
            for ind in range(self.n_indicators):
                hmm = self.hmm_list[ind]
                hmm_test_data = X[:, cable, ind].reshape(-1, 1)
                metrics[0, cable, ind] = hmm.score(hmm_test_data)
                metrics[1, cable, ind] = hmm.aic(hmm_test_data)
                metrics[2, cable, ind] = hmm.bic(hmm_test_data)
        return metrics

    def sweep(self, X_train: np.ndarray, X_valid: np.ndarray, n_states_list: list[int]) -> tuple[np.ndarray, np.ndarray]:
        valid_metrics = []
        train_metrics = []
        for n_states in n_states_list:
            self.n_states = n_states
            self.fit(X_train)
            train_metrics.append(self.score(X_train))
            valid_metrics.append(self.score(X_valid))

        return np.stack(train_metrics), np.stack(valid_metrics)

    def fit(self, X: np.ndarray):
        assert X.shape[-1] == self.n_indicators, "Input data must have the correct number of indicators."
        n_cables = X.shape[1]

        # hmm_train_data = np.vstack(
        #     # (
        #     #     X[:, 0],
        #     #     X[:, 1],
        #     #     X[:, 2],
        #     #     X[:, 3],
        #     # )
        #     (X[:, i] for i in range(n_cables)
        # )

        hmm_train_data = np.vstack([X[:, i] for i in range(n_cables)])
        lens_train = [X.shape[0]] * n_cables

        for ind in range(self.n_indicators):
            hmm = self.instantiate()
            slices = np.array_split(np.arange(X.shape[0]), self.n_states)
            initial_mean = np.array([X[s,:,ind].mean() for s in slices])
            initial_std = np.array([X[s,:,ind].std() for s in slices])
            hmm.means_ = initial_mean.reshape(-1, 1)
            hmm.covars_ = (initial_std ** 2).reshape(-1, 1)
            hmm.fit(hmm_train_data[:, [ind]], lengths=lens_train)
            self.hmm_list[ind] = hmm

    def predict(self, X: np.ndarray, use_viterbi: bool = False, min_samples_to_transition: int = 1, std_factor: float = 1.0) -> np.ndarray:
        n_cables = X.shape[1]
        states = np.zeros((X.shape[0], X.shape[1], self.n_indicators), dtype=int)
        for ind in range(self.n_indicators):
            hmm: GaussianHMM = self.hmm_list[ind]
            for cable in range(n_cables):
                hmm_test_data = X[:, cable, ind]
                if use_viterbi:
                    states_ind = hmm.predict(hmm_test_data.reshape(-1,1))
                else:
                    states_ind = self.forward_predict(hmm_test_data.reshape(-1,1), hmm, min_samples_to_transition, std_factor)
                states[:, cable, ind] = states_ind

        return states
    
    def forward_predict(self, x: np.ndarray, hmm: GaussianHMM, min_samples_to_transition: int, std_factor: float) -> np.ndarray:
        # Create the GMM objects
        gmm_list = []
        for k in range(self.n_states):
            gmm0 = GaussianMixture(n_components=1, covariance_type="full")
            gmm0.fit(np.random.rand(10, hmm.n_features))
            gmm0.means_ = hmm.means_[k].reshape(1, -1)
            gmm0.covariances_ = hmm.covars_[k] * std_factor
            # gmm0.weights_ = [1] #hmm.weights_[k]
            gmm_list.append(gmm0)

        # Compute emission probabilities b(x)
        b = []
        for o in x:
            temp = []
            for k in range(len(gmm_list)):
                temp.append(np.exp(gmm_list[k].score(o.reshape(1, -1))))
            temp = np.array(temp)
            b.append(temp)
        b = np.array(b)

        # Compute alpha using the forward algorithm
        eps = 1e-16
        A = np.round(hmm.transmat_.copy(), 3)
        s = "start"
        pi = np.round(hmm.startprob_.copy(), 3)
        out = []
        last_transition = -1
        for k in range(len(x)):
            bk = b[k, :]
            if s == "start":
                log_alpha = np.log(pi + eps) + np.log(bk + eps)
            else:
                log_alpha_0 = log_alpha.copy()
                for j in range(A.shape[0]):
                    log_alpha[j] = (
                        logsumexp(np.log(A[:, j] + eps) + log_alpha_0) + np.log(bk + eps)[j]
                    )
                    if A[s, j] == 0:
                        log_alpha[j] = -1e10
                    if k - last_transition < min_samples_to_transition and j != s:
                        log_alpha[j] = -1e10
            
            new_s = np.argmax(log_alpha)
            if new_s != s and k - last_transition >= min_samples_to_transition:
                s = new_s
                last_transition = k
            out.append(new_s)

        states = np.array(out)
        return states

# Sweeps

In [ ]:
hmm = HMMConditionIndicator(n_states=7, p_self=0.8, n_indicators=train_ind.shape[2])
# hmm.fit(train_ind)

In [ ]:
valid_cables = [0,1,2]

In [ ]:
# states_sweep = np.arange(2, 9)
# train_metrics, valid_metrics = hmm.sweep(train_ind[:,valid_cables], valid_ind, n_states_list=states_sweep)

In [ ]:
# metric_names = ["Log Likelihood", "AIC", "BIC"]

# for metric in range(3):
#     plt.figure(figsize=(5, 3))
#     for ind in range(train_ind.shape[2]):
#         plt.subplot(1, train_ind.shape[2], ind + 1)
#         plt.title(f"Metric {metric_names[metric]} - Cable Mean - Indicator {ind + 1}")
#         plt.plot(states_sweep, train_metrics[:,metric,:,ind].max(axis=1), marker="x")
#         plt.plot(states_sweep, valid_metrics[:,metric,:,ind].max(axis=1), marker="x")
#     plt.tight_layout()

# Final HMM

In [ ]:
valid_cables = [0,1,2]
hmm = HMMConditionIndicator(n_states=7, p_self=0.99, n_indicators=train_ind.shape[2])
hmm.fit(train_ind[:,valid_cables])

In [ ]:
min_samples_to_transition = 2
std_factor = 1
use_viterbi = False

In [ ]:
train_preds = hmm.predict(train_ind, use_viterbi=use_viterbi, min_samples_to_transition=min_samples_to_transition, std_factor=std_factor)
valid_preds = hmm.predict(valid_ind, use_viterbi=use_viterbi, min_samples_to_transition=min_samples_to_transition, std_factor=std_factor)

In [ ]:
test_preds = hmm.predict(test_ind, use_viterbi=use_viterbi, min_samples_to_transition=min_samples_to_transition, std_factor=std_factor)

In [ ]:
train_score = hmm.score(train_ind)[0].squeeze() / len(train_ind)
valid_score = hmm.score(valid_ind)[0].squeeze() / len(valid_ind)

In [ ]:
test_score = hmm.score(test_ind)[0].squeeze() / len(test_ind)

In [ ]:
cable_str = ["Cable 1", "Cable 2", "Cable 3", "Cable 4"]

plt.figure(figsize=(4,1.5))
plt.plot(cable_str, train_score, alpha=1, label="Train", color='k', marker="x")
plt.plot(cable_str, valid_score, alpha=1, label="Valid", color='blue', marker="x")
plt.ylabel(r"$\frac{\ln p(\mathbf{X}_j | {\theta})}{N}$")
plt.tick_params(axis='both', which='minor', bottom=False, left=True, top=False, right=True)
plt.legend(fontsize=7, loc="lower left")

plt.tight_layout()
plt.savefig("figures_vae_results/hmm_scores.png", dpi=400)



In [ ]:
cable_str = ["Cable 1", "Cable 2", "Cable 3", "Cable 4"]

plt.figure(figsize=(4,1.5))
plt.plot(cable_str, train_score, alpha=0.2, label="Train", color='k', marker="x")
plt.plot(cable_str, valid_score, alpha=0.2, label="Valid", color='blue', marker="x")
plt.plot(cable_str, test_score, alpha=1, label="Test", color='red', marker="x", linestyle="-.")
plt.ylabel(r"$\frac{\ln p(\mathbf{X}_j | {\theta})}{N}$")
plt.tick_params(axis='both', which='minor', bottom=False, left=True, top=False, right=True)
plt.legend(fontsize=7, loc="lower left")

plt.tight_layout()
plt.savefig("figures_vae_results/hmm_scores_test.png", dpi=400)


In [ ]:
np.round(hmm.hmm_list[0].transmat_, 3)

In [ ]:
np.round(hmm.hmm_list[0].means_.flatten(), 3), np.round(np.sqrt(hmm.hmm_list[0].covars_.flatten()), 3)

In [ ]:
normalized_train_preds = train_preds.squeeze() / (hmm.n_states - 1)
normalized_valid_preds = valid_preds.squeeze() / (hmm.n_states - 1)

plt.figure(figsize=(4, 4))
for cable in range(4):
    plt.subplot(2,2,cable+1)
    plt.title(f"Cable {cable+1}")
    plt.plot(train_preds[:,cable] + 1, linestyle="-")
    plt.plot(valid_preds[:,cable] + 1, linestyle="--", color="blue")
    plt.xlabel("n (days)")
    plt.ylabel(r"$s_{%d}[n]$" % (cable + 1))
    plt.legend(["Train", "Validation"], fontsize=6, framealpha=1, loc="lower right")
    plt.ylim(ymin=0.75, ymax=hmm.n_states+0.25)
    plt.xlim(xmin=0)
    plt.yticks(np.arange(1, hmm.n_states + 1));
    plt.xticks(np.arange(0, 65, 10))
    plt.tick_params(axis='both', which='minor', bottom=True, left=False, top=True, right=False)
plt.tight_layout()
plt.savefig("figures_vae_results/hmm_states_per_cable_per_n.png", dpi=400)


In [ ]:
plt.figure(figsize=(4, 4))
for cable in range(4):
    plt.subplot(2,2,cable+1)
    plt.title(f"Cable {cable+1}")
    plt.plot(t_train_gb[:,cable], train_preds[:,cable] + 1, linestyle="-")
    plt.plot(t_valid_gb[:,cable], valid_preds[:,cable] + 1, linestyle="--", color="blue")
    plt.xlabel(r"$\bar{t}_{%d}[n]$" % (cable + 1))
    plt.ylabel(r"$s_{%d}[n]$" % (cable + 1))
    plt.legend(["Train", "Validation"], fontsize=6, framealpha=1, loc="lower right")
    plt.ylim(ymin=0.75, ymax=hmm.n_states+0.25)
    plt.xlim(xmin=0)
    plt.yticks(np.arange(1, hmm.n_states + 1));
    plt.xticks(np.arange(0, 0.6, 0.1))
    plt.tick_params(axis='both', which='minor', bottom=True, left=False, top=True, right=False)
plt.tight_layout()
plt.savefig("figures_vae_results/hmm_states_per_cable_per_t.png", dpi=400)



In [ ]:
plt.figure(figsize=(4, 4))
for cable in range(4):
    plt.subplot(2,2,cable+1)
    plt.title(f"Cable {cable+1}")
    plt.plot(train_preds[:,cable] + 1, linestyle="-", alpha=0.25, linewidth=1.0)
    plt.plot(valid_preds[:,cable] + 1, linestyle="--", color="blue", alpha=0.25, linewidth=1.0)
    plt.plot(test_preds[:,cable] + 1, linestyle="-.", color="red", linewidth=1.5)
    plt.xlabel("n (Days)")
    plt.ylabel(r"$s_{%d}[n]$" % (cable + 1))
    plt.legend(["Train", "Validation", "Test"], fontsize=6, framealpha=1, loc="lower right")
    plt.ylim(ymin=0.75, ymax=hmm.n_states+0.25)
    plt.xlim(xmin=0)
    plt.yticks(np.arange(1, hmm.n_states + 1));
    # plt.xticks(np.arange(0, 0.6, 0.1))
    plt.tick_params(axis='both', which='minor', bottom=True, left=False, top=True, right=False)
plt.tight_layout()
plt.savefig("figures_vae_results/hmm_states_per_cable_per_n_test.png", dpi=400)

In [ ]:
plt.figure(figsize=(4, 4))
for cable in range(4):
    plt.subplot(2,2,cable+1)
    plt.title(f"Cable {cable+1}")
    plt.plot(t_train_gb[:,cable], train_preds[:,cable] + 1, linestyle="-", alpha=0.25, linewidth=1.0)
    plt.plot(t_valid_gb[:,cable], valid_preds[:,cable] + 1, linestyle="--", color="blue", alpha=0.25, linewidth=1.0)
    plt.plot(t_test_gb[:,cable], test_preds[:,cable] + 1, linestyle="-.", color="red", linewidth=1.5)
    plt.xlabel(r"$\bar{t}_{%d}[n]$" % (cable + 1))
    plt.ylabel(r"$s_{%d}[n]$" % (cable + 1))
    plt.legend(["Train", "Validation", "Test"], fontsize=6, framealpha=1, loc="lower right")
    plt.ylim(ymin=0.75, ymax=hmm.n_states+0.25)
    plt.xlim(xmin=0)
    plt.yticks(np.arange(1, hmm.n_states + 1));
    plt.xticks(np.arange(0, 0.6, 0.1))
    plt.tick_params(axis='both', which='minor', bottom=True, left=False, top=True, right=False)
plt.tight_layout()
plt.savefig("figures_vae_results/hmm_states_per_cable_per_t_test.png", dpi=400)



# View states

In [ ]:
means = hmm.hmm_list[0].means_.reshape(-1)
covars = hmm.hmm_list[0].covars_.reshape(-1)
stds = np.sqrt(covars)
states = np.arange(len(means)) + 1  # State 1 to 7

plt.figure(figsize=(4,2))
for i, (m, std) in enumerate(zip(means, stds)):
    y = np.linspace(m - 4*std, m + 4*std, 500)
    y = np.clip(y, a_min=0, a_max=None)  # Ensure y values are non-negative
    pdf = (1 / (std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((y - m) / std)**2)
    pdf_scaled = (pdf / pdf.max()) * 0.4
    plt.plot(states[i] + pdf_scaled, y, color='royalblue', lw=1.5, linestyle='-')
    plt.fill_betweenx(y, states[i], states[i] + pdf_scaled, color='royalblue', alpha=0.3)
    plt.plot([states[i], states[i] + 0.4], [m, m], color='black', linestyle='--', lw=0.5, alpha=0.6)
    plt.scatter(states[i], m, color='k', s=15, zorder=5, label='Mean' if i == 0 else "")

plt.plot(states, means, color="k", alpha=0.5)
plt.ylim(ymin=0)
plt.xticks(states);
plt.tick_params(axis='both', which='minor', bottom=False, left=True, top=False, right=True)
plt.xlabel("HMM State")
plt.ylabel("Health Indicator")
plt.tight_layout()
plt.savefig("figures_vae_results/hmm_state_distributions.png", dpi=400)
